# 🛡️ AI Social Listening - Chạy Gradio trên Google Colab

Notebook này **clone code từ GitHub về Colab**, **tự động xóa code cũ nếu tồn tại**, cài dependencies, rồi **chạy Dashboard Gradio với public URL** (dùng Gradio share tunnel).

### ⚠️ Bước chuẩn bị

1. **Push code lên GitHub** (repo hiện tại chưa có commit nào — phải commit & push trước khi clone được):
   ```bash
   git add .
   git commit -m "Gradio App"
   git remote add origin <URL_REPO>
   git push -u origin main
   ```
2. Điền `GITHUB_REPO` ở ô bên dưới.
3. Chạy tuần tự các ô (▶) từ trên xuống.

In [ ]:
# ================= CONFIG =================
# Điền URL repo GitHub chứa dự án (thư mục chính là Gradio_App/):
GITHUB_REPO = "https://github.com/dyno511/Demo_NLP_AnhDuy"

# (Tùy chọn) Muốn clone vào đúng một thư mục, đặt tay; để None thì tự suy từ URL
PROJECT_DIR = None
print("GITHUB_REPO =", GITHUB_REPO)

In [ ]:
import os
import shutil

# ---- 1. Xác định thư mục ----
_name = GITHUB_REPO.rstrip("/").split("/")[-1].replace(".git", "")
REPO_DIR = PROJECT_DIR if PROJECT_DIR else f"/content/{_name}"
PROJECT_PATH = os.path.join(REPO_DIR, "Gradio_App")

print("[*] Clone repo vào:  ", REPO_DIR)
print("[*] Dự án chính là:  ", PROJECT_PATH)

# ---- 2. Tự động xóa code cũ nếu tồn tại ----
if os.path.exists(REPO_DIR):
    print(f"[*] Phát hiện code cũ tại {REPO_DIR} -> Đang XÓA...")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print("[*] Đã xóa code cũ.")
else:
    print("[*] Không có code cũ — clone mới hoàn toàn.")

# ---- 3. Clone code mới ----
%cd /content
!git clone "{GITHUB_REPO}" "{REPO_DIR}"
print()
print("[*] Nội dung sau khi clone:")
!ls -la "{REPO_DIR}"
print("[*] Nội dung thư mục dự án Gradio_App:")
!ls -la "{PROJECT_PATH}"

### 📦 Cài đặt dependencies

Colab đã có sẵn `torch`/`transformers` (PhoBERT) nên chỉ cài các package cần thêm: `gradio`, `pandas`, `schedule`, `python-dotenv`, `requests`, `facebook-scraper`, `lxml_html_clean`.

In [ ]:
%cd "{PROJECT_PATH}"
!pip install -q -U uvicorn "gradio>=4" pandas schedule python-dotenv requests "facebook-scraper" lxml_html_clean

# QUAN TRỌNG: pin websockets xuống bản tương thích với uvicorn của Colab.
# websockets >= 15 đã gỡ 'ServerProtocol' khỏi websockets.server -> uvicorn crash:
#   ImportError: cannot import name 'ServerProtocol' (kèm theo lỗi "port already in use" giả mạo)
!pip install -q -U "websockets>=10.0,<15"

# Xác nhận phép fix hoạt động: bước này phải in ra 'ServerProtocol OK'
!python -c "import uvicorn, websockets; print('uvicorn', uvicorn.__version__, '| websockets', websockets.__version__); from websockets.server import ServerProtocol; print('ServerProtocol OK')"

# Kiểm tra giới thiệu các gói quan trọng
import gradio, pandas, schedule, dotenv, requests
print("gradio =", gradio.__version__)
print("pandas =", pandas.__version__)
print("OK - Dependencies sẵn sàng.")

### 🚀 Chạy Gradio với PUBLIC URL

Chạy cell này sẽ:
- Load hệ thống (lần đầu Colab sẽ **tải model PhoBERT** từ HuggingFace, khoảng vài trăm MB — chỉ mất thời gian lần đầu)
- Mở Gradio Dashboard với **public URL dạng `https://xxxx.gradio.live`** (dùng Gradio share tunnel)

Cell sẽ chạy **không dừng lại** cho tới khi hết thời gian phiên (runtime). Tìm dòng log:
```
Running on public URL: https://xxxxxx.gradio.live
```
mở link đó là bạn truy cập được Web Dashboard từ bất kỳ đâu. (Dùng `Ctrl/⌘ + Click` hoặc nhấn vào mã QR hiển thị bên dưới log.)

In [ ]:
%cd "{PROJECT_PATH}"
# Dọn tiến trình Gradio còn sót lại (tránh lỗi "port already in use")
!pkill -f "gradio_app.py" 2>/dev/null || true

# Tự chọn một cổng TRỐNG để tránh xung đột port trên Colab
import socket
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.bind(("0.0.0.0", 0))
FREE_PORT = sock.getsockname()[1]
sock.close()
print("Auto chọn cổng trống:", FREE_PORT)

!python gradio_app.py --share --server-name 0.0.0.0 --server-port {FREE_PORT}

### 💡 Ghi chú

- **Ảnh model PhoBERT**: `gradio_app.py` tự tải model `wonrax/phobert-base-vietnamese-sentiment` khi khởi động; nếu tải quá lâu bạn có thể chạy trước cell sau để tải 1 lần:
  ```python
  from transformers import pipeline, AutoTokenizer
  pipeline("text-classification", model="wonrax/phobert-base-vietnamese-sentiment", tokenizer="wonrax/phobert-base-vietnamese-sentiment")
  ```
- **Cấu hình (.env)**: file `.env` không nằm trong git (đã ignore). Colab sẽ dùng giá trị mặc định trong `config.py`. Muốn đổi tổ chức mục tiêu / kênh Telegram, sửa trong tab **⚙️ Cấu Hình Hệ Thống** trên Web UI.
- **Phiên Colab tắt thì public URL chết**: URL chỉ hoạt động khi cell đang chạy.
- Muốn chạy lại từ đầu: chạy lại 1 lần **Runtime → Restart session**.